# 🎧 Higgs Audio v3: Google Colab Benchmark & Playground (TTS + STT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/higgs_colab_benchmark.ipynb)

Этот блокнот позволяет запускать и тестировать модели **Higgs Audio v3** от Boson AI на облачных GPU в Google Colab (NVIDIA T4 / L4 / A100 / V100):
1. **Higgs TTS 3 (4B)** () — Синтез русской речи, управление эмоциями/стилем, клонирование голоса.
2. **Higgs STT v3 (2.68B)** () — Распознавание русской речи (Whisper-Large-v3 + Qwen3).
3. **Бенчмарк производительности** — Замер времени загрузки, времени генерации, RTF (Real-Time Factor) и потребления VRAM для сравнения с локальным запуском на Apple Silicon M1.

## 1. Проверка GPU и установка зависимостей

In [ ]:
# Проверяем доступность NVIDIA GPU
!nvidia-smi

# Устанавливаем совместимые версии библиотек
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.51.0" "accelerate>=0.26.0" soundfile librosa jiwer sentencepiece huggingface_hub


In [ ]:
import os
import time
import torch
import soundfile as sf
import numpy as np
from pathlib import Path
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Используемое устройство: {device}, тип данных: {dtype}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Higgs STT v3 (Speech-to-Text) Benchmark
Загрузка модели  (2.68B) и распознавание русской аудиозаписи.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

STT_MODEL_ID = "bosonai/higgs-audio-v3-stt"

print("Загрузка вспомогательного кода STT...")
stt_code_dir = snapshot_download(STT_MODEL_ID, allow_patterns=["*.py"])
import sys
if stt_code_dir not in sys.path:
    sys.path.insert(0, stt_code_dir)

from transcribe import transcribe

print("Загрузка весов STT модели на GPU...")
t0 = time.perf_counter()
stt_tokenizer = AutoTokenizer.from_pretrained(STT_MODEL_ID, trust_remote_code=True)
stt_model = AutoModel.from_pretrained(
    STT_MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    attn_implementation="eager",
    device_map="auto"
)
stt_model.eval()
stt_load_time = time.perf_counter() - t0

print(f" STT модель загружена за {stt_load_time:.2f} сек")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB (пик: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB)")


In [ ]:
# Создаем или скачиваем тестовое русское аудио
sample_audio_path = "sample_stt.wav"

if not os.path.exists(sample_audio_path):
    sr = 16000
    t = np.linspace(0, 5, 5 * sr, endpoint=False)
    sf.write(sample_audio_path, (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32), sr)

print(f"Распознавание {sample_audio_path}...")
audio_data, sr = sf.read(sample_audio_path)
duration_sec = len(audio_data) / sr

t0 = time.perf_counter()
transcript = transcribe(stt_model, stt_tokenizer, sample_audio_path, sample_rate=16000)
stt_proc_time = time.perf_counter() - t0
stt_rtf = stt_proc_time / duration_sec

print(f"
 Результат транскрипции:
{transcript}")
print(f"
 Метрики STT:")
print(f" - Длительность аудио: {duration_sec:.2f} сек")
print(f" - Время обработки:   {stt_proc_time:.2f} сек")
print(f" - RTF (Proc/Audio):   {stt_rtf:.2f}x (меньше 1.0x = быстрее реального времени)")


## 3. Higgs TTS 3 (Text-to-Speech) Benchmark & Generation
Синтез русской речи с поддержкой эмоциональных тегов на модели .

In [ ]:
TTS_MODEL_ID = "bosonai/higgs-tts-3-4b"
print(f"Загрузка токенизатора и модели TTS: {TTS_MODEL_ID}...")

from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.perf_counter()
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_ID, trust_remote_code=True)
tts_model = AutoModelForCausalLM.from_pretrained(
    TTS_MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    device_map="auto"
)
tts_model.eval()
tts_load_time = time.perf_counter() - t0

print(f" TTS модель загружена за {tts_load_time:.2f} сек")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")


In [ ]:
# 3.1 Базовый синтез русской речи
text_basic = "Сегодня мы проверяем работу системы синтеза речи Higgs Audio на облачном сервере с GPU."
print(f"Генерация базовой речи: "{text_basic}"")


In [ ]:
# 3.2 Синтез с тегами эмоций и просодии
text_controls = "<|emotion:contentment|><|prosody:speed_slow|>Начнём спокойно и внимательно. <|prosody:pause|> Теперь голос становится выразительнее. <|emotion:enthusiasm|><|prosody:expressive_high|>Это важная и радостная проверка! <|prosody:long_pause|><|style:whispering|>А теперь тихое завершение."
print(f"Генерация с тегами управления:
{text_controls}")


## 4. Сравнительная таблица производительности (Apple M1 vs NVIDIA GPU)

In [ ]:
import pandas as pd

comparison_data = {
    "Платформа": ["Apple Silicon M1 (16GB) - Локально", "Google Colab (CUDA)"],
    "STT Устройство": ["Metal GPU (MPS FP16)", f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"} ({dtype})"],
    "STT RTF (меньше = лучше)": ["1.40x", f"{stt_rtf:.2f}x" if "stt_rtf" in locals() else "N/A"],
    "TTS Basic RTF": ["7.02x", "Измеряется в Colab"],
    "TTS Controls RTF": ["12.61x", "Измеряется в Colab"]
}

df = pd.DataFrame(comparison_data)
display(df)
